# Preparing Data for Modelling

Reads a filtered interim dataset from notebook `02`, imputes missing `vehicle_start_year`, and builds the person-interval counting-process frame for survival models in R.

## What this step does

1. Load `personal_users_filtered.csv` (or professional equivalent).
2. `DataProcessor.apply_feature_engineering`:
   - interval grid from each user's **first → last activity date** (14-day default);
   - empty mid-history intervals kept as explicit inactivity;
   - covariates aggregated per interval, with lookback windows scoped **per user**;
   - `churn_triggered` max'd per interval, then `shift(-1)` → `churn_triggered_adjusted` (features at *t*, event at *t→t+1*);
   - each user's **incomplete last interval dropped** after that shift.
3. Save feature CSVs under `Data/final` for `Coding/R/survival_modelling.R`.

Registration-based features (`account_tenure_days`, `onboarding_delay_days`) appear when `registered_date` was joined in notebook `02`.

Dataset splitting, resampling, modelling, and evaluation stay in R.


In [1]:
# Locate the Coding project root so the src package imports resolve consistently
import os
import polars as pl
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent, parent / "Coding")
    if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir()
)
os.chdir(PROJECT_ROOT)
PROJECT_ROOT

PosixPath('/Users/tomas.p/Desktop/Survival_Analysis_Thesis/Coding')

In [2]:
from src.constants import paths_to_files_and_folders as const
from src.data_processing import DataProcessor

In [3]:
# Filtered interim files from notebook 02
path_to_personal_filtered = const.PATH_TO_INTERIM_DATA / "personal_users_filtered.csv"
path_to_professional_filtered = const.PATH_TO_INTERIM_DATA / "professional_users_filtered.csv"

# Final feature frames consumed by R (and optional 28-day sensitivity run)
SAVE_TO_14_PERSONAL = (
    const.PATH_TO_FINAL_DATA / "features_personal_14_day_intervals_new_features.csv"
)
SAVE_TO_28_PERSONAL = (
    const.PATH_TO_FINAL_DATA / "features_personal_28_day_intervals_new_features.csv"
)

In [4]:
personal = pl.read_csv(path_to_personal_filtered)
# Expect churn label + registration date from notebook 02
assert "churn_triggered" in personal.columns
assert "registered_date" in personal.columns
personal.shape

(129432, 14)

In [5]:
# DataProcessor applies imputation and feature engineering to the complete segment dataset
processor = DataProcessor(personal)

In [6]:
# Primary modelling frame: 14-day intervals, 3 lookback lags (covers 0–56 day windows)
personal_features_14 = processor.apply_feature_engineering(
    df=personal,
    interval_in_days=14,
    save_file_to=SAVE_TO_14_PERSONAL,
    lookback_periods=(1, 2, 3),
)

# Optional sensitivity: coarser 28-day intervals (one lag covers the prior window)
personal_features_28 = processor.apply_feature_engineering(
    df=personal,
    interval_in_days=28,
    save_file_to=SAVE_TO_28_PERSONAL,
    lookback_periods=(1,),
)

print(
    f"14-day: {personal_features_14.height} rows, "
    f"{personal_features_14['user_id'].n_unique()} users, "
    f"{personal_features_14.width} columns"
)
print(
    f"churn rate (interval-level): "
    f"{personal_features_14['churn_triggered_adjusted'].mean():.3f}"
)
assert "registered_date" not in personal_features_14.columns  # intermediate, not kept
for col in (
    "account_tenure_days",
    "onboarding_delay_days",
    "weekend_share_0_28_days",
    "usage_concentration",
    "n_new_vehicles_0_28_days",
    "days_since_last_new_vehicle",
    "vehicle_mean_mileage_ordinal",
    "vehicle_age_sd_overall",
    "churn_triggered_adjusted",
):
    assert col in personal_features_14.columns, f"missing {col}"
print("OK: expected modelling columns present")


/Users/tomas.p/Desktop/Survival_Analysis_Thesis/Coding/src/data_processing.py:245: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_with_intervals = df.join_asof(


14-day: 50311 rows, 1945 users, 34 columns
churn rate (interval-level): 0.019
OK: expected modelling columns present


/Users/tomas.p/Desktop/Survival_Analysis_Thesis/Coding/src/data_processing.py:245: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_with_intervals = df.join_asof(


In [7]:
personal_features_14.columns

['user_id',
 'interval_start',
 'interval_end',
 'prop_clear_0_56',
 'prop_coding_0_56',
 'prop_history_screen_0_56',
 'prop_live_data_0_56',
 'prop_oca_0_56',
 'prop_scan_0_56',
 'prop_main_0_56',
 'prop_main_drift_0_28_vs_28_56',
 'n_sessions_0_28_days',
 'sessions_intensity_drift_0_28_vs_28_56_days',
 'actions_per_session_0_28_days',
 'actions_per_session_intensity_drift_0_28_vs_28_56_days',
 'recency',
 'weekend_share_0_28_days',
 'usage_concentration',
 'vehicle_mean_age_overall',
 'vehicle_age_sd_overall',
 'prop_in_prod',
 'vehicle_mean_mileage_ordinal',
 'n_new_vehicles_0_28_days',
 'days_since_last_new_vehicle',
 'overall_prop_vehicle_make_audi',
 'overall_prop_vehicle_make_skoda',
 'overall_prop_vehicle_make_volkswagen',
 'churn_triggered_adjusted',
 'active_flag_0_56_days',
 'CV_gap_0_56_days',
 'mean_gap_0_56_days',
 'sd_gap_0_56_days',
 'account_tenure_days',
 'onboarding_delay_days']